# File 4 — Prep data for final Tri-1 analysis file

This replaces the prior 16-variable cap. It derives the same parity and education indicators, retains `birth_id`, and carries **every `t1_*` variable** into one frozen analysis file. Final analytic restrictions remain in Stata so the sample-selection audit is visible in one place.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, numpy as np, pandas as pd
INPUT_PARQUET = '/content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_allvars_v3.parquet'
INPUT_CSV = '/content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_allvars_v3.csv'
OUTPUT_ROOT = '/content/drive/MyDrive/plos-update-v3/4-analysis'
DATA_DIR = os.path.join(OUTPUT_ROOT,'data'); STATA_DIR = os.path.join(OUTPUT_ROOT,'stata'); TABLE_DIR = os.path.join(OUTPUT_ROOT,'tables')
for d in [DATA_DIR,STATA_DIR,TABLE_DIR]: os.makedirs(d, exist_ok=True)
df = pd.read_parquet(INPUT_PARQUET) if os.path.exists(INPUT_PARQUET) else pd.read_csv(INPUT_CSV, low_memory=False)
print('Rows loaded:', f'{len(df):,}')

Mounted at /content/drive
Rows loaded: 164,977


In [ ]:
required = ['birth_id','county_fips','birth_year','gest_age_weeks','conception_quarter','birthweight_g','infant_male','maternal_age','maternal_race_broad','married','prenatal_by5','live_birth_order','maternal_education_years']
missing = [c for c in required if c not in df.columns]
if missing: raise KeyError('Missing required fields: ' + ', '.join(missing))
t1cols = sorted([c for c in df.columns if c.lower().startswith('t1_')])
if not t1cols: raise KeyError('No T1 variables found.')

def indicator(condition, valid):
    condition = pd.Series(condition, index=df.index).astype('boolean')
    valid = pd.Series(valid, index=df.index).fillna(False).astype(bool)
    out = pd.Series(np.nan, index=df.index, dtype=float)
    out.loc[valid] = condition.loc[valid].fillna(False).astype(float)
    return out
parity_valid = df['live_birth_order'].notna()
df['parity2'] = indicator(df['live_birth_order'].eq(2), parity_valid)
df['parity3plus'] = indicator(df['live_birth_order']>=3, parity_valid)
edu_valid = df['maternal_education_years'].notna()
df['meduc_hs'] = indicator(df['maternal_education_years'].eq(12), edu_valid)
df['meduc_gt_hs'] = indicator(df['maternal_education_years']>12, edu_valid)

core = ['birth_id','county_fips','birth_year','gest_age_weeks','conception_quarter','birthweight_g','infant_male','maternal_age','maternal_race_broad','married','prenatal_by5','parity2','parity3plus','meduc_hs','meduc_gt_hs']
final_cols = core + [c for c in t1cols if c not in core]
final_df = df[final_cols].copy()
assert final_df['birth_id'].is_unique
assert final_df['county_fips'].notna().all()
print('T1 fields carried to final analysis file:')
print([c for c in final_df.columns if c.lower().startswith('t1_')])

T1 fields carried to final analysis file:
['t1_allobs', 't1_any_county_primary_ge10', 't1_any_ge10', 't1_any_pws_ge10', 't1_anyimp', 't1_days', 't1_end', 't1_impdays', 't1_max_finished_water', 't1_mean_complete', 't1_mean_observed', 't1_nquarters', 't1_obsdays', 't1_obsfrac', 't1_start']


In [ ]:
qa_rows = []
for c in [x for x in final_df.columns if x.lower().startswith('t1_')]:
    s = pd.to_numeric(final_df[c], errors='coerce')
    qa_rows.append({'variable':c,'n_nonmissing':int(s.notna().sum()),'mean':s.mean(),'min':s.min(),'max':s.max()})
qa = pd.DataFrame(qa_rows)
display(qa)

OUT_PARQUET = os.path.join(DATA_DIR,'analysis_ready_T1_allvars_v3.parquet')
OUT_CSV = os.path.join(DATA_DIR,'analysis_ready_T1_allvars_v3.csv')
OUT_DTA = os.path.join(STATA_DIR,'analysis_ready_T1_allvars_v3.dta')
final_df.to_parquet(OUT_PARQUET,index=False)
final_df.to_csv(OUT_CSV,index=False)
final_df.to_stata(OUT_DTA,write_index=False,version=118)
qa.to_csv(os.path.join(TABLE_DIR,'all_T1_variable_QA.csv'),index=False)
print('Saved final all-variable analysis file:', OUT_CSV)

,variable,n_nonmissing,mean,min,max
0,t1_allobs,164977,6.752456e-01,0.000000e+00,1.000000e+00
1,t1_any_county_primary_ge10,164977,6.973093e-02,0.000000e+00,1.000000e+00
2,t1_any_ge10,164977,1.403105e-01,0.000000e+00,1.000000e+00
3,t1_any_pws_ge10,164977,1.403105e-01,0.000000e+00,1.000000e+00
4,t1_anyimp,164977,3.247544e-01,0.000000e+00,1.000000e+00
5,t1_days,164977,9.800000e+01,9.800000e+01,9.800000e+01
6,t1_end,164977,4.988838e+17,4.186080e+17,5.825952e+17
7,t1_impdays,164977,2.164073e+01,0.000000e+00,9.800000e+01
8,t1_max_finished_water,145922,6.916103e+00,0.000000e+00,4.900000e+01
9,t1_mean_complete,164977,3.467369e+00,0.000000e+00,3.913673e+01


Saved final all-variable analysis file: /content/drive/MyDrive/plos-update-v3/4-analysis/data/analysis_ready_T1_allvars_v3.csv
